In [ ]:
# Libraries for data handling and model training
import numpy as np
import pandas as pd
from datasets import Dataset, Features, Value, Array2D

In [ ]:
# Just for context: Logs include a few metadata lines starting with '#', followed by numeric rows.
# Each numeric row layout: frame, steering, throttle, width, height, then 25*25 depth values
paths = [
    "cleaned_60.csv",
    "cleaned_61.csv",
    "cleaned_62.csv",
    "circuit_1.csv",
    "simple_strip.csv",
    "maze.csv",
    "dead_end.csv",
    "more_data.csv",
    "walls_with_gaps.csv",
    "circle_loop_conference.csv",
    "more_circle.csv",
    "hard_course.csv"
]
need_to_be_converted = ["walls_with_gaps.csv", "circle_loop_conference.csv", "more_circle.csv", "hard_course.csv"]

H, W = 25, 25
N_DEPTH = H * W

depth_cols = [f"d{i}" for i in range(N_DEPTH)]
col_names = ["frame", "steering", "throttle", "width", "height"] + depth_cols

dfs = []
for p in paths:
    tmp = pd.read_csv(
        p,
        comment="#", # ignore metadata + commented header lines
        header=None, # numeric data has no real header row
        names=col_names # assign stable column names
    )
    if p in need_to_be_converted:
        tmp["steering"] = tmp["steering"].astype(int) / 1000.0
        tmp["throttle"] = tmp["throttle"].astype(int) / 1000.0
    dfs.append(tmp)
df = pd.concat(dfs, ignore_index=True)

In [ ]:
df

In [ ]:
import numpy as np
from datasets import Dataset, Features, Value, Array2D

H, W = 25, 25
pixel_cols = [f"d{i}" for i in range(H * W)]

# Extract pixel block
pixels = df[pixel_cols].to_numpy(dtype=np.int32)
images = pixels.reshape(-1, H, W)

# Build dataset directly from dict (no object column)
dataset = Dataset.from_dict({
    "image": images,
    "steering": df["steering"].astype(np.float32).to_numpy(),
    "throttle": df["throttle"].astype(np.float32).to_numpy(),
})

# Define schema explicitly
features = Features({
    "image": Array2D(shape=(H, W), dtype="int32"),
    "steering": Value("float32"),
    "throttle": Value("float32"),
})

dataset = dataset.cast(features)

In [ ]:
dataset.push_to_hub("regularpooria/tinynav_depth_camera_circuits")